📝 **Author:** Amirhossein Heydari - 📧 **Email:** <amirhosseinheydari78@gmail.com> - 📍 **Origin:** [mr-pylin/pytorch-workshop](https://github.com/mr-pylin/pytorch-workshop)

---


**Table of contents**<a id='toc0_'></a>    
- [Dependencies](#toc1_)    
- [Classification of GTZAN Dataset](#toc2_)    
  - [Utility Function to Store Metrics](#toc2_1_)    
  - [Hyperparameters](#toc2_2_)    
  - [Pre-Processing](#toc2_3_)    
    - [Custom Dataset Class](#toc2_3_1_)    
    - [Custom Subset Class](#toc2_3_2_)    
    - [Define Transforms and Initialize Dataset](#toc2_3_3_)    
    - [Generate Weird HTML Code for Playing Audio Files](#toc2_3_4_)    
    - [Plot Mel-Spectrograms of Audio Files](#toc2_3_5_)    
  - [Custom MLP model](#toc2_4_)    
    - [Initialize the Model](#toc2_4_1_)    
  - [Train and Validation Loop](#toc2_5_)    
    - [Analyze Loss and Accuracy over Epochs](#toc2_5_1_)    
  - [Test Loop](#toc2_6_)    
    - [Plot Top_1 Confusion Matrix](#toc2_6_1_)    
    - [Classification Report](#toc2_6_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependencies](#toc0_)


In [1]:
from collections import Counter
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchaudio
from IPython.display import HTML, Audio, display
from sklearn.metrics import classification_report
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchaudio.transforms import AmplitudeToDB, MelSpectrogram, Resample, Vol
from torchinfo import summary
from torchmetrics.classification import MulticlassAccuracy, MulticlassConfusionMatrix
from torchvision.transforms import Normalize

In [ ]:
# list available audio backends (empty? check requirements.txt for PySoundFile)
print(f"torchaudio.list_audio_backends(): {torchaudio.list_audio_backends()}")

In [3]:
# set a seed for deterministic results
seed = 42
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# check if cuda is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# log
print(f"device: {device}")

In [5]:
# update paths as needed based on your project structure
DATASET_DIR = r"../../../../datasets/GTZAN/genres_original/"
LOG_DIR = r"./results"

# <a id='toc2_'></a>[Classification of GTZAN Dataset](#toc0_)


## <a id='toc2_1_'></a>[Utility Function to Store Metrics](#toc0_)


In [6]:
class MetricsLogger:
    def __init__(
        self,
        train_val_file: str = f"{LOG_DIR}/train_val_metrics.csv",
        test_file: str = f"{LOG_DIR}/test_metrics.csv",
        confusion_matrix_file: str = f"{LOG_DIR}/test_top_1_confusion_matrix.csv",
        test_top_k_acc: int = 5,
        lr_precision: str = ".6f",
        loss_precision: str = "7.5f",
        acc_precision: str = ".3f",
    ):
        self.train_val_file = train_val_file
        self.test_file = test_file
        self.confusion_matrix_file = confusion_matrix_file
        self.test_top_k_acc = test_top_k_acc
        self.lr_precision = lr_precision
        self.loss_precision = loss_precision
        self.acc_precision = acc_precision

        # initialize csv files with headers
        self._initialize_file(
            self.train_val_file,
            "epoch,lr,train_loss,train_acc,val_loss,val_acc\n",
        )
        self._initialize_file(
            self.test_file,
            f"test_loss,{','.join(f'test_top_{i+1}_acc' for i in range(test_top_k_acc))}\n",
        )

    def _initialize_file(self, file_path: str, header: str) -> None:

        # create directory if doesn't exist
        Path(file_path).parent.mkdir(parents=True, exist_ok=True)

        with open(file_path, mode="w") as file:
            file.write(header)

    def log_train_val(
        self, epoch: str, lr: float, train_loss: float, train_acc: float, val_loss: float, val_acc: float
    ) -> None:
        with open(self.train_val_file, mode="a") as file:
            file.write(
                f"{epoch},{lr:{self.lr_precision}},{train_loss:{self.loss_precision}},{train_acc:{self.acc_precision}},{val_loss:{self.loss_precision}},{val_acc:{self.acc_precision}}\n"
            )

    def log_test(self, test_loss: float, *test_top_k_acc: float) -> None:

        if len(test_top_k_acc) != self.test_top_k_acc:
            raise ValueError(f"expected {self.test_top_k_acc} test accuracies, but got {len(test_top_k_acc)}.")

        with open(self.test_file, mode="a") as file:
            file.write(
                f"{test_loss:{self.loss_precision}},{','.join(f'{x:{self.acc_precision}}' for x in test_top_k_acc)}\n"
            )

    def log_confusion_matrix(self, cm: torch.Tensor, labels: list[str]) -> None:

        if cm.dim() != 2:
            raise ValueError("confusion matrix must be a 2D tensor.")

        self._initialize_file(
            self.confusion_matrix_file,
            f",{",".join([f'pred_{label}' for label in labels])}\n",
        )

        max_length_label = max(map(len, labels))

        with open(self.confusion_matrix_file, mode="a") as file:
            for true_label_idx, true_label in enumerate(labels):
                row = [f"true_{true_label:<{max_length_label}}"] + [
                    f"{cm[true_label_idx, pred_idx]}" for pred_idx in range(cm.shape[1])
                ]
                file.write(",".join(row) + "\n")

In [7]:
test_top_k_acc = 5
logger = MetricsLogger(test_top_k_acc=test_top_k_acc)

## <a id='toc2_2_'></a>[Hyperparameters](#toc0_)


In [8]:
TRAIN_BATCH_SIZE = 32
BASE_DROPOUT_RATE = 0.5
LEARNING_RATE = 0.001
EPOCHS = 25

## <a id='toc2_3_'></a>[Pre-Processing](#toc0_)


### <a id='toc2_3_1_'></a>[Custom Dataset Class](#toc0_)


In [9]:
class AudioFolder(Dataset):
    def __init__(self, root_dir: str, transform=None, cache=True):
        self.transform = transform
        self.cache = cache
        self.cache_dict = {}  # to store cached transformed audio

        # target duration in samples (e.g., for 30.013333333333332 seconds at 22050 Hz)
        # 30.013333333333332 was carefully obtained using python: <(audio_length / sample_rate)>
        self.valid_audio_length = int(30.013333333333332 * 22050)

        # get all .wav files from the directory and subdirectories
        self.data = glob(f"{root_dir}/**/*.wav")

        # extract labels from the directory names (assuming labels are folder names)
        self.targets = [Path(l).parent.name for l in self.data]

        # find unique labels
        self.classes = sorted(set(self.targets))

        # encode labels into numbers
        self.class_to_idx = {l: i for i, l in enumerate(self.classes)}

        # convert labels to indices [encode labels]
        self.targets = [self.class_to_idx[label] for label in self.targets]

    def __len__(self):
        return len(self.data)

    def _load_audio(self, audio_path: str, raw_audio: bool = False) -> torch.Tensor:
        waveform, sample_rate = torchaudio.load(audio_path)

        # clip waveforms that exceed the valid_audio_length
        if waveform.shape[-1] > self.valid_audio_length:
            waveform = waveform[:, : self.valid_audio_length]

        # pad waveforms that are shorter than the valid_audio_length with 0
        elif waveform.shape[-1] < self.valid_audio_length:
            padding_size = self.valid_audio_length - waveform.shape[-1]
            waveform = F.pad(waveform, (0, padding_size))

        # apply transform if not None
        if self.transform and not raw_audio:
            waveform = self.transform(waveform)

        return waveform

    def __getitem__(self, idx: int, raw_audio: bool = False) -> tuple[torch.Tensor, int]:
        audio_path, label = self.data[idx], self.targets[idx]

        # return raw audio to play in the notebook
        if raw_audio:
            return self._load_audio(audio_path, raw_audio), label

        # each audio file is cached individually to stop calculating mel-spectrograms for each epochs
        if self.cache and audio_path not in self.cache_dict:

            # apply transformations (if not cached already)
            transformed_waveform = self._load_audio(audio_path)
            self.cache_dict[audio_path] = transformed_waveform

        else:

            # retrieve the transformed waveform from cache
            transformed_waveform = self.cache_dict[audio_path]

        return transformed_waveform, label

### <a id='toc2_3_2_'></a>[Custom Subset Class](#toc0_)

- In order to split the whole **dataset** into **train**, **validation**, and **test** sets.


In [10]:
class DatasetSplit:
    def __init__(self, dataset, samples_per_class, proportions):
        self.dataset = dataset
        self.samples_per_class = samples_per_class
        self.proportions = proportions

        # validate proportions
        if sum(proportions) != 1.0:
            raise ValueError("proportions must sum to 1.0")

        self.split_indices = self._create_splits()

    def _create_splits(self):
        total_samples = len(self.dataset)
        num_classes = total_samples // self.samples_per_class

        train_p, validation_p, test_p = self.proportions
        train_size = int(self.samples_per_class * train_p)
        validation_size = int(self.samples_per_class * validation_p)
        test_size = self.samples_per_class - train_size - validation_size

        train_indices = []
        validation_indices = []
        test_indices = []

        for i in range(num_classes):
            start = i * self.samples_per_class
            train_indices.extend(range(start, start + train_size))
            validation_indices.extend(range(start + train_size, start + train_size + validation_size))
            test_indices.extend(range(start + train_size + validation_size, start + self.samples_per_class))

        return {"train": train_indices, "validation": validation_indices, "test": test_indices}

    def get_subset(self, split_name: str):
        if split_name not in self.split_indices:
            raise ValueError(f"invalid split name: {split_name}. choose from 'train', 'validation', 'test'.")
        return Subset(self.dataset, self.split_indices[split_name])

### <a id='toc2_3_3_'></a>[Define Transforms and Initialize Dataset](#toc0_)

📚 **Tutorials**:

- Audio Feature Extractions: [pytorch.org/audio/main/tutorials/audio_feature_extractions_tutorial.html](https://pytorch.org/audio/main/tutorials/audio_feature_extractions_tutorial.html)


In [11]:
# compose transforms [there is no class like <Compose> in `torchvision` here so we use <Sequential> instead!]
transform = nn.Sequential(
    # changes the number of samples per second in the audio
    Resample(orig_freq=22050, new_freq=16000),
    # normalize the loudness (volume) of the audio signal
    Vol(gain=1.0, gain_type="amplitude"),
    # transforms the time-domain signal into a 2D representation (frequency over time)
    MelSpectrogram(sample_rate=16000, n_mels=64, hop_length=1024, n_fft=1024, f_min=0.0, f_max=8000),
    # converts the amplitude (linear scale) into dB, making the range more suitable for input to models
    AmplitudeToDB(stype="power"),
    # standardizing the input features, making them more suitable for neural networks
    # mean and std computed only from the trainset!
    Normalize(mean=[4.2353], std=[14.1284]),
)

In [ ]:
# create a dataset
dataset = AudioFolder(DATASET_DIR, transform=transform, cache=True)

# split dataset into {train, validation, test}
dataset_split = DatasetSplit(dataset, samples_per_class=100, proportions=(0.85, 0.05, 0.1))
trainset = dataset_split.get_subset("train")
validationset = dataset_split.get_subset("validation")
testset = dataset_split.get_subset("test")

# create dataloaders
trainloader = DataLoader(trainset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
validationloader = DataLoader(validationset, batch_size=len(validationset), shuffle=False, num_workers=0)
testloader = DataLoader(testset, batch_size=len(testset), shuffle=False, num_workers=0)

# log
print("dataset:")
print(f"    -> type(dataset.data)    : {type(dataset.data)}")
print(f"    -> type(dataset.targets) : {type(dataset.targets)}")
print(f"    -> dataset.data[0]       : {dataset.data[0]}")
print(f"    -> dataset.targets[0]    : {dataset.targets[0]}")
print(f"    -> dataset[0][0].shape   : {dataset[0][0].shape}")
print(f"    -> dataset[0][0].dtype   : {dataset[0][0].dtype}")
print(f"    -> dataset[0][1]         : {dataset[0][1]}")
print(f"    -> dataset.classes       : {dataset.classes}")
print(f"    -> dataset.class_to_idx  : {dataset.class_to_idx}\n")
print("trainset:")
print(f"    -> len(trainset)              : {len(trainset)}")
print(f"    -> trainset distribution      : {dict(Counter([dataset.targets[idx] for idx in trainset.indices]))}\n")
print("validationset:")
print(f"    -> len(validationset)         : {len(validationset)}")
print(f"    -> validationset distribution : {dict(Counter([dataset.targets[idx] for idx in validationset.indices]))}\n")
print("testset:")
print(f"    -> len(testset)               : {len(testset)}")
print(f"    -> testset distribution       : {dict(Counter([dataset.targets[idx] for idx in testset.indices]))}\n")
print("trainloader:")
print(f"    -> next(iter(trainloader))[0].shape      : {next(iter(trainloader))[0].shape}")
print(f"    -> next(iter(trainloader))[0].dtype      : {next(iter(trainloader))[0].dtype}")
print(f"    -> next(iter(trainloader))[1].shape      : {next(iter(trainloader))[1].shape}")
print(f"    -> next(iter(trainloader))[1].dtype      : {next(iter(trainloader))[1].dtype}\n")
print("validationloader:")
print(f"    -> next(iter(validationloader))[0].shape : {next(iter(validationloader))[0].shape}")
print(f"    -> next(iter(validationloader))[0].dtype : {next(iter(validationloader))[0].dtype}")
print(f"    -> next(iter(validationloader))[1].shape : {next(iter(validationloader))[1].shape}")
print(f"    -> next(iter(validationloader))[1].dtype : {next(iter(validationloader))[1].dtype}\n")
print("testloader:")
print(f"    -> next(iter(testloader))[0].shape       : {next(iter(testloader))[0].shape}")
print(f"    -> next(iter(testloader))[0].dtype       : {next(iter(testloader))[0].dtype}")
print(f"    -> next(iter(testloader))[1].shape       : {next(iter(testloader))[1].shape}")
print(f"    -> next(iter(testloader))[1].dtype       : {next(iter(testloader))[1].dtype}\n")

### <a id='toc2_3_4_'></a>[Generate Weird HTML Code for Playing Audio Files](#toc0_)


In [ ]:
rows = 2
columns = 5

# collect rows of HTML for the table
table_rows = []

# loop through classes and generate HTML for each
row_html = "<tr>"
for class_idx, class_name in enumerate(dataset.classes):
    # get the first sample index for the current class
    sample_index = class_idx * 100  # Assuming consecutive samples per class
    audio_data = dataset.__getitem__(sample_index, raw_audio=True)[0]

    # create HTML for the current column (class title + audio widget)
    cell_html = f"""
    <td style='text-align: center; padding: 5px; width: 120px;'>
        <h5>{class_name} first audio</h5>
        {Audio(audio_data, rate=22050)._repr_html_()}
    </td>
    """
    row_html += cell_html

    # add a new row after every `columns` cells
    if (class_idx + 1) % columns == 0:
        row_html += "</tr>"
        table_rows.append(row_html)
        row_html = "<tr>"

# add remaining row if not complete
if row_html != "<tr>":
    row_html += "</tr>"
    table_rows.append(row_html)

# combine all rows into a table
table_html = f"""
<table style='width: 100%; border-collapse: collapse; table-layout: fixed;'>
    {''.join(table_rows)}
</table>
"""

# display the table
display(HTML(table_html))

### <a id='toc2_3_5_'></a>[Plot Mel-Spectrograms of Audio Files](#toc0_)


In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(14, 3), layout="compressed")

# loop through each class and generate the Mel-spectrogram for the first sample
for idx, class_name in enumerate(dataset.classes):

    # get the first sample index for the current class
    sample_index = idx * 100
    mel_spec_db = dataset[sample_index][0]

    # plot the Mel-spectrogram on the respective subplot
    ax = axes[idx // 5, idx % 5]  # Adjusted for 2 rows and 5 columns
    ax.imshow(mel_spec_db.squeeze(), cmap="inferno", origin="lower", aspect="auto")
    ax.set_title(f"{class_name} first audio")
    # ax.axis("off")

plt.suptitle("Mel-Spectrograms of First Audio Sample for Each Class")
plt.savefig(f"{LOG_DIR}/features_demo.png", format="png", bbox_inches="tight", dpi=72)
plt.show()

## <a id='toc2_4_'></a>[Custom MLP model](#toc0_)


In [15]:
class CustomMLP(nn.Module):
    def __init__(self, input_size: int, num_classes: int):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 512)
        self.dropout1 = nn.Dropout(BASE_DROPOUT_RATE)
        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(BASE_DROPOUT_RATE / 2)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten the input which is a 2D mel-spectrogram.
        x = self.dropout1(F.relu(self.fc1(x)))
        x = self.dropout2(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

### <a id='toc2_4_1_'></a>[Initialize the Model](#toc0_)


In [ ]:
input_size = torch.prod(torch.tensor(dataset[0][0].shape))
num_classes = len(dataset.classes)

# initialize the MLP model
model = CustomMLP(input_size=input_size, num_classes=num_classes).to(device)

# log
model

In [ ]:
summary(model, input_size=(TRAIN_BATCH_SIZE, input_size))

## <a id='toc2_5_'></a>[Train and Validation Loop](#toc0_)


In [ ]:
# initialize the loss function, optimizer, lr scheduler, and accuracy metrics
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, threshold=1e-4)
train_acc = MulticlassAccuracy(num_classes, top_k=1).to(device)
validation_acc = MulticlassAccuracy(num_classes, top_k=1).to(device)

# loss and accuracy placeholders
train_loss_per_epoch = []
train_acc_per_epoch = []
validation_loss_per_epoch = []
validation_acc_per_epoch = []

for epoch in range(EPOCHS):
    # train section
    model.train()
    train_loss = 0.0

    for x, y_true in trainloader:

        # move batch of features and labels to <device>
        x, y_true = x.to(device), y_true.to(device)

        # forward
        y_pred = model(x)

        # loss
        loss = criterion(y_pred, y_true)

        # backward
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # store loss and accuracy per iteration
        train_loss += loss.item() * len(x)
        train_acc.update(y_pred, y_true)

    # store loss and accuracy per epoch
    train_loss_per_epoch.append(train_loss / len(trainset))
    train_acc_per_epoch.append(train_acc.compute().item())
    train_acc.reset()

    # validation section
    model.eval()
    validation_loss = 0.0

    with torch.no_grad():
        for x, y_true in validationloader:

            # move batch of features and labels to <device>
            x, y_true = x.to(device), y_true.to(device)

            # forward
            y_pred = model(x)

            # loss
            loss = criterion(y_pred, y_true)

            # store loss and accuracy per iteration
            validation_loss += loss.item() * len(x)
            validation_acc.update(y_pred, y_true)

    # store loss and accuracy per epoch
    validation_loss_per_epoch.append(validation_loss / len(validationset))
    validation_acc_per_epoch.append(validation_acc.compute().item())
    validation_acc.reset()

    # lr scheduler
    scheduler.step(validation_loss_per_epoch[epoch])

    # store train and validation metrics
    logger.log_train_val(
        epoch=f"{epoch+1:0{len(str(EPOCHS))}}",
        lr=scheduler.get_last_lr()[0],
        train_loss=train_loss_per_epoch[epoch],
        train_acc=train_acc_per_epoch[epoch],
        val_loss=validation_loss_per_epoch[epoch],
        val_acc=validation_acc_per_epoch[epoch],
    )

    # log
    print(
        f"epoch {epoch+1:0{len(str(EPOCHS))}}/{EPOCHS} -> lr: {scheduler.get_last_lr()[0]:.5f} | train[loss: {train_loss_per_epoch[epoch]:9.5f} - accuracy: {train_acc_per_epoch[epoch]*100:5.2f}%]  |  validation[loss: {validation_loss_per_epoch[epoch]:9.5f} - accuracy: {validation_acc_per_epoch[epoch]*100:5.2f}%]"
    )

### <a id='toc2_5_1_'></a>[Analyze Loss and Accuracy over Epochs](#toc0_)


In [ ]:
# plot
fig, ax = plt.subplots(1, 2, figsize=(14, 4), layout="compressed")

ax[0].plot(train_acc_per_epoch, label="Train Accuracy", marker="o", color="blue")
ax[0].plot(validation_acc_per_epoch, label="Validation Accuracy", marker="o", color="orange")
ax[0].set(
    title="Accuracy Over Epochs",
    xlabel="Epoch",
    ylabel="Accuracy",
    xticks=range(EPOCHS),
    yticks=torch.arange(0, 1.1, 0.1),
)
ax[0].legend()
ax[0].grid()

ax[1].plot(train_loss_per_epoch, label="Train Loss", marker="o", color="blue")
ax[1].plot(validation_loss_per_epoch, label="Validation Loss", marker="o", color="orange")
ax[1].set(title="Loss Over Epochs", xlabel="Epoch", ylabel="Loss", xticks=range(EPOCHS))
ax[1].legend()
ax[1].grid()

plt.suptitle("Training and Validation Accuracy and Loss Over Epochs")
plt.savefig(f"{LOG_DIR}/train_val_metrics.svg", format="svg", bbox_inches="tight")
plt.show()

## <a id='toc2_6_'></a>[Test Loop](#toc0_)


In [ ]:
top_k_acc = []
true_labels = []
predictions = []

for k in range(test_top_k_acc):

    model.eval()
    test_acc = MulticlassAccuracy(num_classes, top_k=k + 1).to(device)
    test_loss = 0.0

    with torch.no_grad():
        for x, y_true in testloader:

            # move batch of features and labels to <device>
            x, y_true = x.to(device), y_true.to(device)

            # forward
            y_pred = model(x)

            # loss
            loss = criterion(y_pred, y_true)

            # store loss and accuracy per iteration
            test_loss += loss.item() * len(x)
            test_acc.update(y_pred, y_true)

            # store predictions and true_labels
            if k == 0:
                predictions.extend(y_pred.argmax(dim=1).cpu())
                true_labels.extend(y_true.cpu())

    # store loss and accuracy per epoch
    test_loss /= len(testset)
    test_acc = test_acc.compute().item()
    top_k_acc.append(test_acc)

# log
print(f"test[loss: {test_loss:.5f} | {' - '.join(f'top_{i} acc: {a*100:5.2f}%' for i, a in enumerate(top_k_acc))}")

In [21]:
# store test metrics
logger.log_test(test_loss, *top_k_acc)

In [22]:
predictions = torch.tensor(predictions).to("cpu")
true_labels = torch.tensor(true_labels).to("cpu")

### <a id='toc2_6_1_'></a>[Plot Top_1 Confusion Matrix](#toc0_)


In [23]:
confmat = MulticlassConfusionMatrix(num_classes)
cm = confmat(predictions, true_labels)

In [24]:
# store confusion matrix
logger.log_confusion_matrix(cm, labels=dataset.classes)

In [ ]:
# plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.set(title="Confusion Matrix based on top_1 accuracy")
confmat.plot(ax=ax, cmap="Blues")
cbar = plt.colorbar(ax.images[0], ax=ax)
cbar.set_label("Count", rotation=270, labelpad=15)
plt.show()

### <a id='toc2_6_2_'></a>[Classification Report](#toc0_)

In [ ]:
# top_1 classification report
print(classification_report(true_labels, predictions))